In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [ ]:
uploaded = files.upload()

df = pd.read_csv("water_potability.csv")

print("Dataset shape:", df.shape)
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

df = df.drop_duplicates()

Saving water_potability.csv to water_potability (1).csv
Dataset shape: (3276, 10)


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0



Missing values:
ph                 491
Hardness             0
Solids               0
Chloramines          0
Sulfate            781
Conductivity         0
Organic_carbon       0
Trihalomethanes    162
Turbidity            0
Potability           0
dtype: int64

Duplicate rows: 0


In [ ]:
class_table_before = df["Potability"].value_counts().reset_index()
class_table_before.columns = ["Potability", "Count"]
class_table_before["Potability"] = class_table_before["Potability"].map({
    0: "Not Drinkable",
    1: "Drinkable"
})

print("\nClass distribution before balancing:")
display(class_table_before)


Class distribution before balancing:


,Potability,Count
0,Not Drinkable,1998
1,Drinkable,1278


In [ ]:
X = df.drop("Potability", axis=1)
y = df["Potability"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining set shape:", X_train.shape)
print("Test set shape:", X_test.shape)


Training set shape: (2620, 9)
Test set shape: (656, 9)


In [ ]:
imputer = SimpleImputer(strategy="median")
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X.columns)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imputed), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imputed), columns=X.columns)


In [ ]:
print("\nBefore SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())

class_table_after = pd.Series(y_train_smote).value_counts().reset_index()
class_table_after.columns = ["Potability", "Count"]
class_table_after["Potability"] = class_table_after["Potability"].map({
    0: "Not Drinkable",
    1: "Drinkable"
})

print("\nClass distribution after SMOTE:")
display(class_table_after)


Before SMOTE:
Potability
0    1598
1    1022
Name: count, dtype: int64

After SMOTE:
Potability
0    1598
1    1598
Name: count, dtype: int64

Class distribution after SMOTE:


,Potability,Count
0,Not Drinkable,1598
1,Drinkable,1598


In [ ]:
baseline_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, C=1.0, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7, weights="distance"),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_split=10, random_state=42)
}

ensemble_models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        random_state=42
    )
}

all_models = {**baseline_models, **ensemble_models}

In [ ]:
results = []
confusion_matrices = {}
trained_models = {}

for name, model in all_models.items():
    model.fit(X_train_smote, y_train_smote)
    trained_models[name] = model

    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    model_type = "Baseline" if name in baseline_models else "Ensemble"

    results.append({
        "Model Type": model_type,
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1 Score": round(f1, 4)
    })

    confusion_matrices[name] = cm

    print(f"\nModel: {name} ({model_type})")
    print("Accuracy:", round(acc, 4))
    print("Precision:", round(prec, 4))
    print("Recall:", round(rec, 4))
    print("F1 Score:", round(f1, 4))
    print("Confusion Matrix:")
    print(cm)
    print("-" * 50)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1 Score", ascending=False).reset_index(drop=True)

print("\nModel Comparison Table:")
display(results_df)


Model: Logistic Regression (Baseline)
Accuracy: 0.5274
Precision: 0.4172
Recall: 0.5312
F1 Score: 0.4674
Confusion Matrix:
[[210 190]
 [120 136]]
--------------------------------------------------

Model: KNN (Baseline)
Accuracy: 0.5762
Precision: 0.4613
Recall: 0.5117
F1 Score: 0.4852
Confusion Matrix:
[[247 153]
 [125 131]]
--------------------------------------------------

Model: Decision Tree (Baseline)
Accuracy: 0.5747
Precision: 0.4637
Recall: 0.5742
F1 Score: 0.5131
Confusion Matrix:
[[230 170]
 [109 147]]
--------------------------------------------------

Model: Random Forest (Ensemble)
Accuracy: 0.6585
Precision: 0.5661
Recall: 0.5352
F1 Score: 0.5502
Confusion Matrix:
[[295 105]
 [119 137]]
--------------------------------------------------

Model Comparison Table:


,Model Type,Model,Accuracy,Precision,Recall,F1 Score
0,Ensemble,Random Forest,0.6585,0.5661,0.5352,0.5502
1,Baseline,Decision Tree,0.5747,0.4637,0.5742,0.5131
2,Baseline,KNN,0.5762,0.4613,0.5117,0.4852
3,Baseline,Logistic Regression,0.5274,0.4172,0.5312,0.4674


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in all_models.items():
    pipe = ImbPipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("model", model)
    ])

    f1_scores = cross_val_score(pipe, X, y, cv=cv, scoring="f1")

    model_type = "Baseline" if name in baseline_models else "Ensemble"

    cv_results.append({
        "Model Type": model_type,
        "Model": name,
        "CV F1 Mean": round(f1_scores.mean(), 4),
        "CV F1 Std": round(f1_scores.std(), 4)
    })

cv_results_df = pd.DataFrame(cv_results)
cv_results_df = cv_results_df.sort_values(by="CV F1 Mean", ascending=False).reset_index(drop=True)

print("\nCross-Validation Results:")
display(cv_results_df)


Cross-Validation Results:


,Model Type,Model,CV F1 Mean,CV F1 Std
0,Ensemble,Random Forest,0.5356,0.0182
1,Baseline,KNN,0.5208,0.0155
2,Baseline,Decision Tree,0.5038,0.0269
3,Baseline,Logistic Regression,0.4289,0.0121


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("\nBest Model based on Test F1 Score:", best_model_name)


Best Model based on Test F1 Score: Random Forest


In [ ]:
sample = pd.DataFrame([{
    "ph": 7.2,
    "Hardness": 204.9,
    "Solids": 20791.0,
    "Chloramines": 7.3,
    "Sulfate": 368.0,
    "Conductivity": 564.0,
    "Organic_carbon": 10.4,
    "Trihalomethanes": 86.9,
    "Turbidity": 2.9
}])

sample_imputed = pd.DataFrame(imputer.transform(sample), columns=sample.columns)
sample_scaled = pd.DataFrame(scaler.transform(sample_imputed), columns=sample.columns)

prediction = best_model.predict(sample_scaled)

print("\nPrediction:", prediction[0])
if prediction[0] == 1:
    print("The water is drinkable.")
else:
    print("The water is not drinkable.")


Prediction: 0
The water is not drinkable.


In [ ]:
for model_name, cm in confusion_matrices.items():
    print(f"\nConfusion Matrix for {model_name}:")
    cm_df = pd.DataFrame(
        cm,
        index=["Actual Not Drinkable", "Actual Drinkable"],
        columns=["Predicted Not Drinkable", "Predicted Drinkable"]
    )
    display(cm_df)


Confusion Matrix for Logistic Regression:


,Predicted Not Drinkable,Predicted Drinkable
Actual Not Drinkable,210,190
Actual Drinkable,120,136



Confusion Matrix for KNN:


,Predicted Not Drinkable,Predicted Drinkable
Actual Not Drinkable,247,153
Actual Drinkable,125,131



Confusion Matrix for Decision Tree:


,Predicted Not Drinkable,Predicted Drinkable
Actual Not Drinkable,230,170
Actual Drinkable,109,147



Confusion Matrix for Random Forest:


,Predicted Not Drinkable,Predicted Drinkable
Actual Not Drinkable,295,105
Actual Drinkable,119,137
